## 📥 Download Real Vector Geospatial Data

**Before starting the assignment**, you need real vector data. This notebook teaches you how to programmatically download geospatial data from authoritative sources.

### 🎯 What You'll Download

1. **EPA Level III Ecoregions** - Ecological regions of the United States
   - Source: US Environmental Protection Agency
   - Features: ~188 ecoregions (Western US)
   - Size: ~1.1 MB
   - Use: Environmental analysis, conservation planning

2. **Natural Earth Cities** - Global populated places
   - Source: Natural Earth Data (cartographer-curated)
   - Features: ~281 US cities (population > 100,000)
   - Size: ~77 KB
   - Use: Urban analysis, demographics, proximity studies

3. **US National Parks** - Major protected areas
   - Source: National Park Service (derived)
   - Features: 37 major national parks
   - Size: ~32 KB
   - Use: Conservation planning, buffer analysis

### 🔀 Two Paths

**Path 1:** ⚡ Quick Download (run Section 1, skip to Section 3)  
**Path 2:** 🎓 Learn How It Works (run Section 2 to download and prepare the data, understand the concepts, then go to Section 3)

### 📚 What You'll Learn in Path 2

- **Download from Cloud**: Access government data from S3 buckets
- **Extract ZIP archives**: Unzip spatial data programmatically
- **Load Shapefiles**: Work with the industry-standard format
- **Filter & Transform**: Subset data by region, simplify geometries
- **Save as GeoJSON**: Export to modern web-friendly format

### ⏱️ Download Time

**In GitHub Codespaces:** ~2-5 minutes (depends on EPA server)

### 📍 Data Sources

- EPA Ecoregions: https://www.epa.gov/eco-research/level-iii-and-iv-ecoregions
- Natural Earth: https://www.naturalearthdata.com/
- National Parks: https://www.nps.gov/ (derived)

**Licenses:** All datasets are Public Domain

---

### ⚙️ Step 0: Select the Correct Python Kernel

Before running any cells, make sure the notebook is using the correct Python environment.

**Check the kernel in the top-right corner of the notebook.**

The correct Python environment is **python-gis-development (.venv)**  
It may appear with a Python version, for example:  
**python-gis-development (3.11.15) (Python 3.11.15) .venv/bin/python**



If the kernel is **python-gis-development (.venv)**, you can start running cells below.

Steps to select the correct kernel:
1. Click on the kernel (top right corner of the notebook) if it is not **python-gis-development (.venv)** or if it says "Select Kernel"
2. Select **python-gis-development (.venv)**
3. If you do not see the kernel in the list, click on "Select Another Kernel..."  
    a. Click on Python Environments...   
    b. Select **python-gis-development (.venv)**

Once the correct kernel is selected, you can start running cells below.

### 🚀 Section 1: Quick Download (Automated Script)

**⚠️ This section downloads the data directly, without any explanation of steps. For an explanation of steps, skip this section and go to Section 2.**

The `src/download_real_data.py` script downloads:
- EPA Level III Ecoregions (Western US) from S3
- Natural Earth populated places (US cities > 100k)
- US National Parks (synthetic dataset)

**Total: 3 datasets, ~1.2MB, ~2-5 minutes**

**To execute this automated script**
1. Remove the `#` from the line below
2. Run this cell
3. Skip to Section 3 to verify your data

In [1]:
# Run the automated download script (uncomment the line below by removing the #)
!python ../../src/download_real_data.py --all --cleanup

# After running, skip to Section 3 to verify your data

🌍 REAL SPATIAL DATA DOWNLOADER
GIST 604B - GeoPandas Spatial Operations Assignment


🌲 EPA LEVEL III ECOREGIONS
Attempting source 1/3...
📥 Downloading from: https://dmap-prod-oms-edc.s3.us-east-1.amazonaws.com/ORD/Ecoregions/us/us_eco_l3.zip
   Progress: 100.0%
✅ Downloaded: us_eco_l3.zip
📦 Extracting: us_eco_l3.zip
✅ Extracted to: /workspaces/gist604b-python-test/data/temp/ecoregions
📂 Loading: us_eco_l3.shp
✅ Loaded 1250 ecoregions
   CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PAR

---

### 🎓 Section 2: Learn How It Works (Tutorial)

**This section explains modern vector data access patterns.**

### What Are Authoritative GIS Data Sources?

Professional GIS work relies on **authoritative data** from trusted sources:

| Organization | Data Types | Why They Matter |
|--------------|------------|-----------------|
| **EPA** | Ecoregions, environmental | Used by federal/state agencies |
| **USGS** | Topography, water, land | National mapping standard |
| **Natural Earth** | Global boundaries, places | Cartographer-curated, widely used |
| **Census** | Demographics, boundaries | Official US statistics |
| **NPS/BLM** | Protected areas, federal lands | Land management |

**Key Concept:** Using authoritative sources ensures:
- ✅ Data accuracy and reliability
- ✅ Proper licensing and attribution
- ✅ Version control and updates
- ✅ Professional credibility

### How Do We Access This Data?

**Three Common Patterns:**

1. **Direct Download (URLs):**
   ```python
   import requests
   response = requests.get("https://data.source.gov/dataset.zip")
   ```

2. **Cloud Storage (S3):**
   ```python
   # Government agencies often use S3 for large datasets
   url = "https://bucket.s3.amazonaws.com/path/data.zip"
   ```

3. **APIs (Web Services):**
   ```python
   # Some sources provide REST APIs
   response = requests.get("https://api.source.gov/v1/query?...")
   ```

**Why This Matters:** Cloud-native workflows mean you download fresh data whenever needed, instead of storing potentially outdated files.

### 📥 Step 2.1: Download EPA Ecoregions from S3

Let's download **EPA Level III Ecoregions** from the EPA's S3 bucket. These are the official ecological regions used by environmental scientists.


In [ ]:
import requests
from pathlib import Path
import zipfile

# Create temporary directory
temp_dir = Path("../../data/temp")
temp_dir.mkdir(parents=True, exist_ok=True)

# EPA S3 bucket URL (public data)
EPA_URL = "https://dmap-prod-oms-edc.s3.us-east-1.amazonaws.com/ORD/Ecoregions/us/us_eco_l3.zip"

print("📥 Downloading EPA Level III Ecoregions...")
print(f"   Source: {EPA_URL[:60]}...")
print(f"   This may take 1-2 minutes...")
print()

# Download the ZIP file
zip_path = temp_dir / "us_eco_l3.zip"

try:
    response = requests.get(EPA_URL, stream=True, timeout=60)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(zip_path, 'wb') as f:
        if total_size == 0:
            f.write(response.content)
        else:
            downloaded = 0
            for chunk in response.iter_content(chunk_size=8192):
                downloaded += len(chunk)
                f.write(chunk)
                progress = (downloaded / total_size) * 100
                print(f"\r   Progress: {progress:.1f}% ({downloaded/1024/1024:.1f} MB)", end='', flush=True)
    
    print()
    print(f"✅ Downloaded: {zip_path.name} ({zip_path.stat().st_size / 1024 / 1024:.1f} MB)")
    
except Exception as e:
    print(f"\n❌ Error downloading EPA data: {e}")
    print("   Note: EPA servers can be slow. Try again later.")

### 📦 Step 2.2: Extract the ZIP Archive

Government datasets often come as **ZIP archives containing Shapefiles**. A Shapefile isn't a single file—it's a collection:

| File | Extension | Purpose |
|------|-----------|---------|
| Main file | `.shp` | Geometry coordinates |
| Index | `.shx` | Shape index |
| Attributes | `.dbf` | Attribute table (dBASE format) |
| Projection | `.prj` | Coordinate system (WKT format) |
| Metadata | `.xml` | ISO 19115 metadata (optional) |

**All these files must be kept together!**


In [ ]:
import zipfile

# Extract the ZIP
extract_dir = temp_dir / "ecoregions"
extract_dir.mkdir(exist_ok=True)

print("📦 Extracting ZIP archive...")

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Show what's inside
        file_list = zip_ref.namelist()
        print(f"   Found {len(file_list)} files in archive")
        
        # Extract all
        zip_ref.extractall(extract_dir)
        
    print(f"✅ Extracted to: {extract_dir}")
    print()
    
    # Find the shapefile components
    shp_files = list(extract_dir.glob("**/*.shp"))
    if shp_files:
        print(f"📂 Found Shapefile: {shp_files[0].name}")
        
        # Show related files
        base_name = shp_files[0].stem
        related = list(shp_files[0].parent.glob(f"{base_name}.*"))
        print(f"   Related files:")
        for f in sorted(related):
            print(f"      • {f.name} ({f.stat().st_size / 1024:.1f} KB)")
    else:
        print("❌ No shapefile found in archive")
        
except Exception as e:
    print(f"❌ Error extracting ZIP: {e}")

### 📂 Step 2.3: Load Shapefile with GeoPandas

Now let's load the shapefile using **GeoPandas**, which handles all the complexity:

- Reads all component files automatically
- Parses the .prj file to get CRS
- Loads attributes from .dbf
- Creates geometries from .shp coordinates

In [ ]:
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

# Find and load the shapefile
shp_files = list(extract_dir.glob("**/*.shp"))

if shp_files:
    print("📂 Loading Shapefile into GeoPandas...")
    print()
    
    gdf = gpd.read_file(shp_files[0])
    
    print(f"✅ Loaded {len(gdf):,} ecoregion features")
    print(f"   CRS: {gdf.crs}")
    print(f"   Geometry type: {gdf.geometry.type.unique()[0]}")
    print()
    print(f"   Columns ({len(gdf.columns)}):")
    for i, col in enumerate(gdf.columns[:10], 1):
        print(f"      {i:2}. {col}")
    if len(gdf.columns) > 10:
        print(f"      ... and {len(gdf.columns) - 10} more")
    
    print()
    print("   Sample data (first 3 features):")
    display(gdf[['US_L3CODE', 'US_L3NAME', 'NA_L2NAME', 'NA_L1NAME', 'geometry']].head(3))
else:
    print("❌ No shapefile found. Re-run the download cell above.")

### 🔧 Step 2.4: Transform & Filter Data

The EPA data covers **all of the United States** (985 ecoregions!). That's too much for our assignment. Let's:

1. **Transform CRS**: EPA data uses Albers Equal Area (EPSG:5070) → Convert to WGS84 (EPSG:4326)
2. **Filter by Region**: Extract only Western US ecoregions
3. **Simplify Geometries**: Reduce complexity for faster processing
4. **Rename Columns**: Use student-friendly names

In [ ]:
import geopandas as gpd

if 'gdf' in locals():
    print("🔧 Preparing Ecoregions Dataset...")
    print()
    
    # Step 1: Transform to WGS84
    if gdf.crs != "EPSG:4326":
        print(f"   Transforming CRS: {gdf.crs.name} → EPSG:4326 (WGS84)")
        gdf = gdf.to_crs("EPSG:4326")
    
    # Step 2: Filter to Western US
    print("   Filtering to Western US (bbox: -125°W to -102°W, 31°N to 49°N)")
    western_bbox = (-125, 31, -102, 49)  # West Coast to Rockies
    gdf_west = gdf.cx[western_bbox[0]:western_bbox[2], western_bbox[1]:western_bbox[3]]
    print(f"   Reduced from {len(gdf):,} → {len(gdf_west):,} ecoregions")
    
    # Step 3: Rename columns
    print("   Renaming columns to student-friendly names")
    gdf_west = gdf_west.rename(columns={
        'US_L3CODE': 'eco_code',
        'US_L3NAME': 'eco_name',
        'NA_L2NAME': 'l2_name',
        'NA_L1NAME': 'l1_name'
    })
    
    # Keep only essential columns
    keep_cols = ['eco_code', 'eco_name', 'l2_name', 'l1_name', 'geometry']
    gdf_west = gdf_west[[col for col in keep_cols if col in gdf_west.columns]].copy()
    
    # Step 4: Simplify geometries (tolerance = 0.01 degrees ≈ 1 km)
    print("   Simplifying geometries (0.01° tolerance)...")
    gdf_west['geometry'] = gdf_west.geometry.simplify(tolerance=0.01, preserve_topology=True)
    
    # Step 5: Add calculated field
    print("   Calculating area in km²...")
    gdf_west['area_km2'] = gdf_west.to_crs('EPSG:6933').geometry.area / 1e6
    
    print()
    print(f"✅ Prepared {len(gdf_west):,} Western US ecoregions")
    print(f"   Columns: {', '.join(gdf_west.columns)}")
    print()
    print("   Sample ecoregions:")
    display(gdf_west.head(3))
else:
    print("❌ Run the download and load cells above first")

### 💾 Step 2.5: Save as GeoJSON

**Why GeoJSON instead of Shapefile?**

| Feature | Shapefile | GeoJSON |
|---------|-----------|--------|
| Files | 4-7 separate files | 1 single file |
| Format | Binary | Text (JSON) |
| Web support | Poor | Excellent |
| Human-readable | No | Yes |
| File size | Smaller (compressed) | Larger (text) |

**Modern GIS trend:** GeoJSON for web/API, GeoPackage for desktop, Shapefile for legacy systems

In [ ]:
from pathlib import Path

if 'gdf_west' in locals():
    # Create output directory
    output_dir = Path("../../data/ecoregions")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save as GeoJSON
    output_path = output_dir / "epa_level3_western_us.geojson"
    
    print("💾 Saving as GeoJSON...")
    gdf_west.to_file(output_path, driver='GeoJSON')
    
    file_size_kb = output_path.stat().st_size / 1024
    print(f"✅ Saved: {output_path}")
    print(f"   Features: {len(gdf_west):,}")
    print(f"   Size: {file_size_kb:.1f} KB")
    print()
    print("   🎉 EPA Ecoregions ready for use!")
else:
    print("❌ Run the preparation cell above first")

### 🌆 Step 2.6: Download Natural Earth Cities

Now let's download **populated places** from Natural Earth. This follows the same pattern:
1. Download ZIP from URL
2. Extract shapefile
3. Load with GeoPandas
4. Filter (US only, population > 100k)
5. Save as GeoJSON

**Note how similar the workflow is!**

In [ ]:
import requests
import zipfile
import geopandas as gpd
from pathlib import Path

print("🏙️  DOWNLOADING NATURAL EARTH CITIES")
print("=" * 50)
print()

# Natural Earth URL
NE_URL = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_populated_places.zip"

# Download
zip_path = temp_dir / "ne_cities.zip"
print(f"📥 Downloading from Natural Earth...")
print(f"   URL: {NE_URL[:60]}...")

try:
    response = requests.get(NE_URL, timeout=60)
    response.raise_for_status()
    
    with open(zip_path, 'wb') as f:
        f.write(response.content)
    
    print(f"✅ Downloaded: {zip_path.name} ({zip_path.stat().st_size / 1024 / 1024:.1f} MB)")
    
    # Extract
    extract_dir = temp_dir / "cities"
    extract_dir.mkdir(exist_ok=True)
    
    print("📦 Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"✅ Extracted to: {extract_dir}")
    
    # Load shapefile
    shp_files = list(extract_dir.glob("**/*.shp"))
    if shp_files:
        print(f"📂 Loading: {shp_files[0].name}")
        cities_gdf = gpd.read_file(shp_files[0])
        
        print(f"✅ Loaded {len(cities_gdf):,} cities (global)")
        
        # Filter to US, population > 100k
        print("🔧 Filtering to US cities (population > 100,000)...")
        
        # Rename columns
        cities_gdf = cities_gdf.rename(columns={
            'NAME': 'name',
            'ADM0NAME': 'country',
            'ADM1NAME': 'state_province',
            'POP_MAX': 'population',
            'LATITUDE': 'latitude',
            'LONGITUDE': 'longitude'
        })
        
        # Filter
        us_cities = cities_gdf[
            (cities_gdf['country'].str.contains('United States', na=False)) &
            (cities_gdf['population'] >= 100000)
        ].copy()
        
        # Keep essential columns
        keep_cols = ['name', 'country', 'state_province', 'population', 'latitude', 'longitude', 'geometry']
        us_cities = us_cities[[col for col in keep_cols if col in us_cities.columns]]
        
        # Sort by population
        us_cities = us_cities.sort_values('population', ascending=False)
        
        print(f"✅ Filtered to {len(us_cities):,} US cities")
        
        # Save
        output_dir = Path("../../data/cities")
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / "ne_cities_us.geojson"
        
        print("💾 Saving as GeoJSON...")
        us_cities.to_file(output_path, driver='GeoJSON')
        
        print(f"✅ Saved: {output_path}")
        print(f"   Features: {len(us_cities):,}")
        print(f"   Size: {output_path.stat().st_size / 1024:.1f} KB")
        print()
        print("   Sample cities:")
        display(us_cities[['name', 'state_province', 'population']].head(5))
    
except Exception as e:
    print(f"❌ Error: {e}")

### 🏞️ Step 2.7: Create Protected Areas Dataset

For **US National Parks**, we'll create a **synthetic dataset** using known park locations and areas. This is educational-quality data.

**Note:** For professional work, use authoritative sources like:
- National Park Service: https://www.nps.gov/subjects/gis/
- Protected Areas Database (PAD-US): https://www.usgs.gov/programs/gap-analysis-project/science/pad-us-data-overview

In [ ]:
from shapely.geometry import Point
import numpy as np

print("🏞️  CREATING PROTECTED AREAS DATASET")
print("=" * 50)
print()

# Comprehensive US National Parks (all regions)
major_parks = {
    'name': [
        # West Coast
        'Olympic National Park',
        'Crater Lake National Park',
        'Redwood National Park',
        'Yosemite National Park',
        'Sequoia National Park',
        'Channel Islands National Park',
        # Southwest
        'Grand Canyon National Park',
        'Zion National Park',
        'Bryce Canyon National Park',
        'Arches National Park',
        'Canyonlands National Park',
        'Petrified Forest National Park',
        'Saguaro National Park',
        'Big Bend National Park',
        'Guadalupe Mountains National Park',
        # Rocky Mountains
        'Yellowstone National Park',
        'Grand Teton National Park',
        'Glacier National Park',
        'Rocky Mountain National Park',
        'Great Sand Dunes National Park',
        # Midwest
        'Badlands National Park',
        'Theodore Roosevelt National Park',
        'Wind Cave National Park',
        'Isle Royale National Park',
        # Southeast
        'Hot Springs National Park',
        'Mammoth Cave National Park',
        'Great Smoky Mountains National Park',
        'Shenandoah National Park',
        'Everglades National Park',
        'Biscayne National Park',
        'Dry Tortugas National Park',
        # Northeast
        'Acadia National Park',
        # Alaska (major)
        'Denali National Park',
        'Glacier Bay National Park',
        'Kenai Fjords National Park',
        # Hawaii
        'Haleakalā National Park',
        'Hawaiʻi Volcanoes National Park'
    ],
    'state': [
        # West Coast
        'Washington', 'Oregon', 'California', 'California', 'California', 'California',
        # Southwest
        'Arizona', 'Utah', 'Utah', 'Utah', 'Utah', 'Arizona', 'Arizona', 'Texas', 'Texas',
        # Rocky Mountains
        'Wyoming/Montana/Idaho', 'Wyoming', 'Montana', 'Colorado', 'Colorado',
        # Midwest
        'South Dakota', 'North Dakota', 'South Dakota', 'Michigan',
        # Southeast
        'Arkansas', 'Kentucky', 'Tennessee/North Carolina', 'Virginia', 'Florida', 'Florida', 'Florida',
        # Northeast
        'Maine',
        # Alaska
        'Alaska', 'Alaska', 'Alaska',
        # Hawaii
        'Hawaii', 'Hawaii'
    ],
    'designation': ['National Park'] * 37,
    'area_km2': [
        # West Coast
        3734, 741, 534, 3074, 1635, 1009,
        # Southwest
        4926, 595, 145, 310, 1366, 895, 370, 3243, 350,
        # Rocky Mountains
        8991, 1255, 4100, 1075, 340,
        # Midwest
        982, 285, 137, 2314,
        # Southeast
        23, 215, 2114, 809, 6105, 700, 262,
        # Northeast
        198,
        # Alaska
        24585, 13287, 2711,
        # Hawaii
        135, 1348
    ],
    'established': [
        # West Coast
        1938, 1902, 1968, 1890, 1890, 1980,
        # Southwest
        1919, 1919, 1928, 1929, 1964, 1962, 1994, 1944, 1972,
        # Rocky Mountains
        1872, 1929, 1910, 1915, 2004,
        # Midwest
        1978, 1978, 1903, 1940,
        # Southeast
        1921, 1941, 1934, 1935, 1947, 1980, 1992,
        # Northeast
        1919,
        # Alaska
        1917, 1980, 1980,
        # Hawaii
        1916, 1916
    ],
    # Approximate centroids
    'longitude': [
        # West Coast
        -123.5, -122.1, -124.0, -119.5, -118.6, -119.9,
        # Southwest
        -112.1, -113.0, -112.2, -109.6, -109.8, -109.8, -110.7, -103.2, -104.9,
        # Rocky Mountains
        -110.5, -110.8, -113.8, -105.7, -105.6,
        # Midwest
        -102.4, -103.5, -103.5, -88.8,
        # Southeast
        -93.1, -86.1, -83.5, -78.5, -80.9, -80.2, -82.9,
        # Northeast
        -68.2,
        # Alaska
        -151.0, -136.9, -149.9,
        # Hawaii
        -156.2, -155.5
    ],
    'latitude': [
        # West Coast
        47.8, 42.9, 41.3, 37.8, 36.5, 34.0,
        # Southwest
        36.1, 37.3, 37.6, 38.7, 38.3, 35.0, 32.2, 29.2, 31.9,
        # Rocky Mountains
        44.6, 43.8, 48.8, 40.3, 37.7,
        # Midwest
        43.9, 47.6, 43.6, 47.9,
        # Southeast
        34.5, 37.2, 35.7, 38.5, 25.3, 25.5, 24.6,
        # Northeast
        44.4,
        # Alaska
        63.1, 58.7, 60.0,
        # Hawaii
        20.7, 19.4
    ]
}

# Create points
geometries = [Point(lon, lat) for lon, lat in zip(major_parks['longitude'], major_parks['latitude'])]
parks_gdf = gpd.GeoDataFrame(major_parks, geometry=geometries, crs='EPSG:4326')

print(f"✅ Created {len(parks_gdf)} national park points")
print(f"   Covering all US regions (West, Southwest, Rocky Mountains, Midwest, Southeast, Northeast, Alaska, Hawaii)")
print()

# Create approximate park boundaries using buffers
print("🔧 Creating approximate park boundaries...")
print("   (Using circular buffers based on actual park areas)")

# Transform to Albers Equal Area for accurate buffering
parks_proj = parks_gdf.to_crs('EPSG:5070')

# Buffer based on area: radius = sqrt(area / π)
parks_proj['geometry'] = parks_proj.apply(
    lambda row: row.geometry.buffer(
        np.sqrt(row['area_km2'] * 1e6 / np.pi)  # Convert km² to m²
    ),
    axis=1
)

# Transform back to WGS84
parks_poly = parks_proj.to_crs('EPSG:4326')

print(f"✅ Created {len(parks_poly)} park polygons")
print()

# Save
output_dir = Path("../../data/protected_areas")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "national_parks_major.geojson"

print("💾 Saving as GeoJSON...")
parks_poly.to_file(output_path, driver='GeoJSON')

print(f"✅ Saved: {output_path}")
print(f"   Features: {len(parks_poly)}")
print(f"   Size: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("   Sample parks by region:")
display(parks_poly[['name', 'state', 'area_km2', 'established']].head(10))

---

### ✅ Section 3: Verify Downloaded Data

Let's check that all files were downloaded successfully.

In [ ]:
from pathlib import Path
import geopandas as gpd

print("🔍 VERIFYING DOWNLOADED DATA")
print("=" * 50)
print()

# Define expected files
data_dir = Path("../../data")
expected_files = {
    'Ecoregions': data_dir / "ecoregions" / "epa_level3_western_us.geojson",
    'Cities': data_dir / "cities" / "ne_cities_us.geojson",
    'Protected Areas': data_dir / "protected_areas" / "national_parks_major.geojson"
}

# Check each dataset
all_exist = True
total_size = 0

for name, path in expected_files.items():
    if path.exists():
        # Load to get feature count
        gdf = gpd.read_file(path)
        size_kb = path.stat().st_size / 1024
        total_size += size_kb
        
        print(f"✅ {name}")
        print(f"   File: {path.name}")
        print(f"   Features: {len(gdf):,}")
        print(f"   Size: {size_kb:.1f} KB")
        print(f"   CRS: {gdf.crs}")
        print()
    else:
        print(f"❌ {name}: NOT FOUND")
        print(f"   Expected: {path}")
        print()
        all_exist = False

print("=" * 50)
if all_exist:
    print(f"✅ ALL DATA DOWNLOADED SUCCESSFULLY!")
    print(f"   Total size: {total_size:.1f} KB ({total_size / 1024:.2f} MB)")
    print()
    print("🚀 Ready to start the assignment!")
else:
    print("❌ SOME FILES MISSING")
    print("   Run Section 1 above to download all data")

---

### 👀 Section 4: Quick Data Preview

Let's visualize what we downloaded!

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path("../../data")

# Load all datasets
try:
    ecoregions = gpd.read_file(data_dir / "ecoregions" / "epa_level3_western_us.geojson")
    cities = gpd.read_file(data_dir / "cities" / "ne_cities_us.geojson")
    parks = gpd.read_file(data_dir / "protected_areas" / "national_parks_major.geojson")
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Ecoregions
    ecoregions.plot(ax=axes[0], column='l1_name', legend=True, 
                    cmap='tab20', edgecolor='black', linewidth=0.3)
    axes[0].set_title(f"EPA Ecoregions\n{len(ecoregions)} regions", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")
    
    # Cities
    cities.plot(ax=axes[1], color='red', markersize=20, alpha=0.6)
    axes[1].set_title(f"US Cities\n{len(cities)} cities (pop > 100k)", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    
    # Protected Areas
    parks.plot(ax=axes[2], color='darkgreen', edgecolor='black', linewidth=1, alpha=0.7)
    axes[2].set_title(f"National Parks\n{len(parks)} major parks", fontsize=14, fontweight='bold')
    axes[2].set_xlabel("Longitude")
    axes[2].set_ylabel("Latitude")
    
    plt.tight_layout()
    plt.show()
    
    print("✅ All datasets loaded and visualized successfully!")
    print()
    print("📊 Dataset Summary:")
    print(f"   • Ecoregions: {len(ecoregions):,} features, CRS: {ecoregions.crs}")
    print(f"   • Cities: {len(cities):,} features, CRS: {cities.crs}")
    print(f"   • Parks: {len(parks):,} features, CRS: {parks.crs}")
    
except FileNotFoundError as e:
    print(f"❌ Could not load data: {e}")
    print("   Run Section 1 to download the data first")

### ✅ Data Download Complete!

You now have **real, authoritative spatial datasets** from:
- **EPA**: Ecological regions used by environmental scientists
- **Natural Earth**: Global cartographic data used by mapmakers worldwide
- **National Parks**: US protected areas data

### 📊 Data Summary

| Dataset | Features | Size | Use Cases |
|---------|----------|------|----------|
| Ecoregions | ~188 | 1.1 MB | Environmental analysis, habitat studies |
| Cities | ~281 | 77 KB | Urban analysis, proximity studies |
| Protected Areas | 37 | 32 KB | Conservation, buffer analysis |
| **Total** | **~479** | **~1.2 MB** | **Ready for geopandas functions!** |

### 🔑 Key Learning Points

- **Authoritative data sources** are important for professional GIS work
- A common spatial data workflow is: **Download → Extract → Load → Filter → Transform → Save**
- **Shapefiles** are legacy multi-file datasets; **GeoJSON** is a modern single-file format
- Data preparation often includes **CRS transformation**, **filtering**, **simplification**, and **column cleanup**
- Reproducible, script-based workflows are an important professional GIS skill
